# 🚀 Getting Started with TorchTPU: LoRA Fine-Tuning Gemma on Cloud TPU v5e

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/1VIQuHE6fh0GRacNcuFcLX5Gn4I0jlmx9?resourcekey=0-nC4YYb3p6X8ZVPeAE4Asxw)

Welcome to the **TorchTPU DevLabs Training Workshop**!

In this notebook, you will learn how to fine-tune a Gemma model on Google Cloud TPUs using standard, idiomatic PyTorch. With TorchTPU, your code looks and feels exactly like standard PyTorch—you simply set `device = torch.device('tpu')` and run.

### What We'll Cover:
1. **Device Setup**: Initializing the TPU device directly through `torch.device('tpu')`.
2. **Open Dataset**: Loading the `yahma/alpaca-cleaned` instruction dataset from Hugging Face / Kaggle.
3. **Model Loading & Auth**: Authenticating with Hugging Face via Colab Secrets (`HF_TOKEN`) and loading `google/gemma-2-2b-it` in native `bfloat16` with activation checkpointing.
4. **LoRA Fine-Tuning**: Attaching low-rank adapters ($r=16$) to Gemma with PEFT, keeping peak HBM usage safely under the **16 GB** limit of a single TPU v5e chip.
5. **TPU Performance Best Practices**:
   - **Static Padding**: Setting fixed sequence lengths (`padding='max_length'`) so the compiler builds the computation graph once on step 1.
   - **Microbatch Gradient Accumulation**: Keeping Gemma-2's 256k-vocab logit head within 16 GB HBM while preserving effective batch size.
   - **Device-Side Logging**: Accumulating loss tensors directly on the TPU to avoid pausing the accelerator on every step.
6. **Verification & Inference**: Running interactive generation with your fine-tuned model directly on TPU.

---
## Step 1: Environment Setup & TorchTPU Verification

Google Colab's TPU runtime comes pre-installed with `torch`, `torch_tpu`, `transformers`, `datasets`, and `accelerate`. Let's install `peft` for LoRA adapter support.

In [ ]:
# Install PEFT (transformers, datasets, accelerate, and torch_tpu are pre-installed on Colab TPU)
!pip install -q peft

In [ ]:
import os
import torch
import torch.nn as nn

# =====================================================================
# TorchTPU integrates natively into PyTorch.
# You interact with TPUs using standard PyTorch device APIs.
# =====================================================================

print(f"PyTorch version: {torch.__version__}")

# Select the Cloud TPU accelerator
device = torch.device("tpu")
print(f"Using device: {device}")

# Quick sanity test: run a bfloat16 matrix multiplication directly on TPU
test_x = torch.ones((1024, 1024), dtype=torch.bfloat16, device=device)
test_y = test_x @ test_x
print(f"Output shape: {test_y.shape}, device: {test_y.device}, dtype: {test_y.dtype}")


---
## Step 2: Load Open Kaggle / Instruction Dataset

We use the popular **`yahma/alpaca-cleaned`** instruction-following dataset (a widely used, open, clean fork of Stanford Alpaca available on Kaggle & Hugging Face).

Each sample contains an `instruction`, an optional `input`, and the target `output`.

In [ ]:
from datasets import load_dataset

# Load the open instruction dataset (no gated login required)
print("Downloading and loading open instruction dataset...")
dataset = load_dataset("yahma/alpaca-cleaned", split="train[:2000]")
print(f"Loaded {len(dataset)} examples for this workshop.")

# Inspect a sample record
sample = dataset[0]
print("Sample record:")
print(f"Instruction: {sample['instruction']}")
print(f"Output: {sample['output']}")


---
## Step 3: Authenticate & Load Gemma Model and Tokenizer

> [!IMPORTANT]
> **Hugging Face Access for Gemma**: `google/gemma-2-2b-it` is a gated repository. Before running the cell below:
> 1. Accept the Gemma license on [huggingface.co/google/gemma-2-2b-it](https://huggingface.co/google/gemma-2-2b-it).
> 2. Add your Hugging Face Read token to **Colab Secrets** (🔑 icon in the left sidebar) named `HF_TOKEN` and toggle **Notebook access** on.

> **TorchTPU Best Practice**: Always load weights in **`torch.bfloat16`**, and enable **gradient checkpointing** so Gemma-2's 26 layers + 256,000-token vocabulary logits fit comfortably inside a single 16 GB TPU v5e chip.

In [ ]:
import os
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM

# Authenticate via Colab Secrets (HF_TOKEN), environment variable, or interactive prompt
hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        hf_token = None

if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("Authenticated with Hugging Face via HF_TOKEN.")
else:
    print("HF_TOKEN not found in Colab Secrets; prompting for Hugging Face login...")
    login(add_to_git_credential=False)

model_id = os.environ.get("GEMMA_MODEL_ID", "google/gemma-2-2b-it")

print(f"Loading tokenizer for {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading base model in bfloat16...")
# =====================================================================
# Cloud TPUs compute natively in bfloat16 at full hardware peak FLOPS.
# Gradient checkpointing discards intermediate layer activations during
# forward and recomputes them in backward so peak HBM stays < 14 GB on v5e-1.
# =====================================================================
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
)
base_model.config.use_cache = False
base_model.gradient_checkpointing_enable()
base_model.enable_input_require_grads()

# Move model to TPU
base_model = base_model.to(device)
print(f"Model loaded and placed on: {device}")

---
## Step 4: Configure LoRA (Parameter-Efficient Fine-Tuning)

Why LoRA + Activation Checkpointing on TPU v5e (16 GB HBM)?
- `google/gemma-2-2b-it` (2.6B parameters) uses **~5.2 GB** of `bfloat16` weights.
- Full fine-tuning requires gradients + FP32 AdamW states for all parameters (**~31–42 GB**), exceeding a single 16 GB v5e chip.
- Even with LoRA, Gemma-2's large `256,000`-token vocabulary means materializing softcapped `float32` cross-entropy logits at `batch_size=4, seq_len=512` spikes to **~30.8 GB HBM**!
- Combining **LoRA** (20.8M trainable params across attention & MLP projections, `<0.8%` of total weights), **gradient checkpointing**, and **microbatching (`MICRO_BATCH_SIZE=1, GRAD_ACCUM_STEPS=4`)** keeps peak TPU HBM around **~13–14 GB**—fitting cleanly inside a single free Colab TPU v5e!

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,                    # Low-rank dimension
    lora_alpha=32,           # Scaling factor (alpha / r = 2.0)
    lora_dropout=0.05,       # Regularization
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

# Wrap base model with PEFT
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

# Ensure the newly created LoRA parameters are also in bfloat16 and on TPU
model = model.to(device=device, dtype=torch.bfloat16)


---
## Step 5: Tokenization with STATIC Shapes (TorchTPU Golden Rule #1)

> [!IMPORTANT]
> **Why Static Shapes on TPU?**
> The OpenXLA compiler generates an optimized machine binary for each unique tensor shape `(micro_batch_size, sequence_length)`.
> If every batch has a different length (dynamic padding), **XLA will pause and recompile the entire graph on every single step**!
>
> By enforcing `padding="max_length"`, `max_length=512`, and `drop_last=True`, every microbatch shape is guaranteed to be fixed `(1, 512)` forever, while `GRAD_ACCUM_STEPS=4` gives an effective batch size of `4` sequences (`2,048` tokens/step) within 16 GB HBM.

In [ ]:
MAX_SEQ_LENGTH = 512
MICRO_BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 4  # Effective batch size = 1 * 4 = 4 sequences (2,048 tokens/step)

def format_and_tokenize(batch):
    # Format into Gemma instruction prompt format:
    # <start_of_turn>user
    # {instruction}
    # {input}<end_of_turn>
    # <start_of_turn>model
    # {output}<end_of_turn>
    formatted_texts = []
    for inst, inp, out in zip(batch["instruction"], batch["input"], batch["output"]):
        user_content = inst if not inp else f"{inst}\n\nContext:\n{inp}"
        text = f"<start_of_turn>user\n{user_content}<end_of_turn>\n<start_of_turn>model\n{out}<end_of_turn>"
        formatted_texts.append(text)

    # =================================================================
    # TPU Performance Best Practice: Use padding='max_length'!
    # Fixed batch shapes mean the graph compiles once on step 1 and never recompiles.
    # =================================================================
    encodings = tokenizer(
        formatted_texts,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding="max_length",
        return_tensors="pt",
    )

    # Mask padding tokens in labels with -100 so they are ignored by cross-entropy loss
    labels = encodings["input_ids"].clone()
    labels[labels == tokenizer.pad_token_id] = -100
    encodings["labels"] = labels

    return encodings

print("Tokenizing dataset with static sequence length...")
tokenized_dataset = dataset.map(
    format_and_tokenize,
    batched=True,
    batch_size=100,
    remove_columns=dataset.column_names,
)
tokenized_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# Create standard PyTorch DataLoader (num_workers=0 avoids fork() warnings after PJRT init)
from torch.utils.data import DataLoader
train_loader = DataLoader(tokenized_dataset, batch_size=MICRO_BATCH_SIZE, shuffle=True, drop_last=True, num_workers=0)
print(
    f"Microbatches ready: {len(train_loader)} "
    f"(Microbatch shape: [{MICRO_BATCH_SIZE}, {MAX_SEQ_LENGTH}], "
    f"Effective batch: {MICRO_BATCH_SIZE * GRAD_ACCUM_STEPS} seqs = {MICRO_BATCH_SIZE * GRAD_ACCUM_STEPS * MAX_SEQ_LENGTH} tokens/step)"
)

---
## Step 6: Training Loop and Host-Device Synchronization

> [!IMPORTANT]
> **Minimizing Host-Device Synchronization**
>
> Like GPUs and other accelerators, the host CPU enqueues operations asynchronously while the TPU executes them in a pipelined fashion.
>
> Calling `.item()` or transferring scalars to CPU on every batch forces an immediate host synchronization: the host blocks, the TPU pipeline flushes, and the accelerator sits idle until the transfer finishes.
>
> **Best Practice**: Accumulate loss tensors on the TPU device and only transfer average metrics to host CPU periodically (e.g., every 10 or 20 steps).

In [ ]:
import time

# Optimize only the trainable LoRA adapter parameters
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=2e-4, weight_decay=0.01)

model.train()
model.config.use_cache = False
NUM_STEPS = 40
LOG_INTERVAL = 10
TOKENS_PER_STEP = MICRO_BATCH_SIZE * GRAD_ACCUM_STEPS * MAX_SEQ_LENGTH

if hasattr(torch.accelerator, "reset_peak_memory_stats"):
    torch.accelerator.reset_peak_memory_stats()

print(f"Starting LoRA fine-tuning on {device} for {NUM_STEPS} optimizer steps ({TOKENS_PER_STEP} tokens/step)...")
start_time = time.time()
window_start = time.time()

# Device-side loss accumulator in float32 (stays on TPU between log intervals)
running_loss_device = torch.zeros((), device=device, dtype=torch.float32)
opt_step = 0
micro_step = 0
optimizer.zero_grad(set_to_none=True)

for batch in train_loader:
    micro_step += 1
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)

    outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
    scaled_loss = outputs.loss / GRAD_ACCUM_STEPS
    scaled_loss.backward()

    # Accumulate loss on TPU without calling .item()
    running_loss_device += scaled_loss.detach().float()

    if micro_step % GRAD_ACCUM_STEPS == 0:
        opt_step += 1
        nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)

        if opt_step == 1:
            # Synchronize once after Step 1 so steady-state throughput excludes initial XLA compilation
            first_loss = running_loss_device.item()
            running_loss_device.zero_()
            step1_sec = time.time() - window_start
            print(f"Step [001/{NUM_STEPS}] (Warmup + XLA Compile) | Loss: {first_loss:.4f} | Time: {step1_sec:.2f}s")
            window_start = time.time()
        elif opt_step % LOG_INTERVAL == 0:
            steps_in_window = (LOG_INTERVAL - 1) if opt_step == LOG_INTERVAL else LOG_INTERVAL
            avg_loss = (running_loss_device / steps_in_window).item()
            running_loss_device.zero_()
            elapsed = time.time() - window_start
            tokens_per_sec = (steps_in_window * TOKENS_PER_STEP) / elapsed
            print(
                f"Step [{opt_step:03d}/{NUM_STEPS}] | Loss: {avg_loss:.4f} | "
                f"Throughput: {tokens_per_sec:.1f} tokens/s | Latency: {elapsed / steps_in_window * 1000:.1f} ms/step"
            )
            window_start = time.time()

        if opt_step >= NUM_STEPS:
            break

torch.accelerator.synchronize()
total_time = time.time() - start_time
peak_hbm_gib = torch.accelerator.max_memory_allocated() / (1024 ** 3)
print(f"\nTraining complete in {total_time:.2f} seconds ({opt_step} steps) | Peak TPU HBM: {peak_hbm_gib:.2f} GiB (fits 16 GiB v5e-1!)")

---
## Step 7: Inference & Qualitative Verification

Now let's test our fine-tuned LoRA model on an unseen instruction to see the output.

In [ ]:
model.eval()
model.config.use_cache = True  # Re-enable KV caching for autoregressive generation

prompt_instruction = "Explain why Cloud TPUs are well-suited for large matrix multiplications in simple terms."
formatted_prompt = f"<start_of_turn>user\n{prompt_instruction}<end_of_turn>\n<start_of_turn>model\n"

inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)

print(f"Prompt:\n{prompt_instruction}\n")
print("Generating response on TPU...")
with torch.no_grad():
    output_tokens = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.pad_token_id,
    )

generated_text = tokenizer.decode(output_tokens[0], skip_special_tokens=True)
print("=" * 60)
print("Generated Response:")
print("=" * 60)
print(generated_text)

---
## Step 8: Saving LoRA Weights & Graduating to TorchTitan

### Saving LoRA Checkpoint
Because we used PEFT, saving only writes the small adapter weights (~40 MB in `bfloat16`) rather than the multi-gigabyte base model:

In [ ]:
save_dir = "./gemma_lora_tpu_checkpoint"
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print(f"Saved lightweight LoRA checkpoint to {save_dir}!")
!ls -lh {save_dir}


### 🎓 Where to Go From Here: Graduating to TorchTitan

In this lab, you mastered single-chip fine-tuning on a Cloud TPU v5e.

When your models and datasets scale to **billions of parameters and hundreds or thousands of accelerator chips**, you don't write custom distributed communication loops by hand. Instead, you use **TorchTitan on TPU** (`google-pytorch/torchtitan`):

```bash
# Example launching TorchTitan Qwen3-8B on a TPU slice with SimpleFSDP:
torchrun --nproc_per_node=8 -m torchtitan.experiments.tpu.train \
    --module=torchtitan.experiments.tpu.qwen3 \
    --config=qwen3_8b \
    --parallelism.use_simple_fsdp \
    --parallelism.enable_tpu_topo_mesh \
    --compile.optimizer_compile=in_graph
```

**What TorchTitan does automatically with TorchTPU:**
1. **Topology-Aware `DeviceMesh` & `SimpleFSDP`**: Native PyTorch `DTensor` collectives (`all_gather`, `reduce_scatter`) mapped onto the TPU Inter-Chip Interconnect (ICI).
2. **SparseCore Collective Offload** *(on TPU v5p, Trillium v6e, and Ironwood v7x)*: FSDP communications run on dedicated SparseCores, leaving TensorCores free for dense matrix math.
3. **Whole-Step Graph Compilation**: Forward pass, backward pass, and optimizer update (`TpuGraphTrainer` with `--compile.optimizer_compile=in_graph`) lower into a unified XLA executable, achieving **~51% MFU** on Qwen3-8B at pod scale.

Check out the open-source repositories:
- [TorchTPU GitHub](https://github.com/google-pytorch/torch_tpu)
- [TorchTitan on TPU GitHub](https://github.com/google-pytorch/torchtitan)